In [ ]:
# ==========================================
# 1. MOUNT GOOGLE DRIVE & SETUP
# ==========================================
# MODIFIED FROM: phase1-5-viirs-ntl-median.ipynb
# CHANGES:
#   - Input: prediction_points.csv instead of DHS GEE asset
#   - VIIRS year: 2025 instead of 2022
#   - Output keyed on PointID, no NTL_Class (inference only)

from google.colab import drive
import os
import ee
import pandas as pd
import numpy as np
import time

drive.mount('/content/drive')

try:
    ee.Initialize(project='integrated-hawk-485001-k3')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='integrated-hawk-485001-k3')

print('GEE initialized.')

In [ ]:
# ==========================================
# 2. LOAD PREDICTION POINTS AND BUILD GEE FEATURE COLLECTION
# ==========================================
# CHANGED from original:
#   - Was: ee.FeatureCollection from GEE asset PH_DHS_GPS
#   - Now: load prediction_points.csv, build FeatureCollection
#     from Latitude/Longitude with PointID as the key.
#   - Buffer logic is identical to training:
#       Urban_Rural == 'U' (NCR) -> 2,000m
#       Urban_Rural == 'R'       -> 5,000m

POINTS_CSV = '/content/drive/MyDrive/Datamaraws_2025_Data/prediction_points.csv'

df_points = pd.read_csv(POINTS_CSV)
print(f'Prediction points loaded : {len(df_points)}')
print(f'Columns                  : {list(df_points.columns)}')
print(f'\nUrban_Rural distribution:')
print(df_points['Urban_Rural'].value_counts().to_string())
print(f'\nProvince distribution:')
print(df_points['Province'].value_counts().to_string())

# Build GEE FeatureCollection
print('\nBuilding GEE FeatureCollection from lat/lon...')
features = []
for _, row in df_points.iterrows():
    pt       = ee.Geometry.Point([float(row['Longitude']),
                                  float(row['Latitude'])])
    radius   = 2000 if str(row['Urban_Rural']).upper() == 'U' \
               else 5000
    buffered = pt.buffer(radius).bounds()

    feat = ee.Feature(buffered, {
        'PointID'     : int(row['PointID']),
        'Province'    : str(row['Province']),
        'Municipality': str(row['Municipality']),
        'Urban_Rural' : str(row['Urban_Rural']),
        'source'      : str(row['source'])
    })
    features.append(feat)

fc_points = ee.FeatureCollection(features)

# Sanity check
n = fc_points.size().getInfo()
print(f'FeatureCollection size: {n}')
assert n == len(df_points), \
    f'Mismatch: {n} in GEE vs {len(df_points)} in CSV'

sample = fc_points.first().getInfo()
print(f'Sample properties: {sample["properties"]}')
assert 'PointID' in sample['properties'], \
    'PointID missing from features'
print('\n\u2713 FeatureCollection ready')

In [ ]:
# ==========================================
# 3. VIIRS PROCESSING
# ==========================================
# CHANGED: filterDate now 2025 instead of 2022.
# Everything else identical to training version:
#   Collection : NOAA/VIIRS/001/VNP46A1
#   Band       : DNB_At_Sensor_Radiance_500m (nW/cm2/sr)
#   Reducer    : annual median

print('Building VIIRS 2025 annual median composite...')

viirs_2025_median = (
    ee.ImageCollection('NOAA/VIIRS/001/VNP46A1')
    .filterDate('2025-01-01', '2025-12-31')
    .select('DNB_At_Sensor_Radiance_500m')
    .median()
)

# Verify composite has data at first point
sample_val = viirs_2025_median.reduceRegion(
    reducer  = ee.Reducer.mean(),
    geometry = fc_points.first().geometry(),
    scale    = 500
).getInfo()
print(f'Sample VIIRS 2025 value at first point: {sample_val}')
print('\n\u2713 VIIRS 2025 composite ready')

In [ ]:
# ==========================================
# 4. REDUCE REGIONS AND EXPORT TO DRIVE
# ==========================================
# CHANGED:
#   - collection is fc_points (from prediction_points.csv)
#   - propertySelectors: PointID, Province, Municipality,
#     Urban_Rural, source, median
#   - Output folder: Datamaraws_2025_Data
#   - Output file  : viirs_ntl_2025_prediction_points.csv

OUTPUT_DIR    = '/content/drive/MyDrive/Datamaraws_2025_Data'
EXPORT_DESC   = 'viirs_ntl_2025_prediction_points'
EXPORT_FOLDER = 'Datamaraws_2025_Data'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Submitting GEE export task...')
print('Estimated time: 5-15 minutes depending on GEE queue.')

viirs_reduced = viirs_2025_median.reduceRegions(
    collection = fc_points,
    reducer    = ee.Reducer.median(),
    scale      = 500    # Native VIIRS resolution
)

# Retain only needed columns to keep file small
viirs_export = viirs_reduced.select(
    propertySelectors = [
        'PointID', 'Province', 'Municipality',
        'Urban_Rural', 'source', 'median'
    ],
    retainGeometry = False
)

export_task = ee.batch.Export.table.toDrive(
    collection     = viirs_export,
    description    = EXPORT_DESC,
    folder         = EXPORT_FOLDER,
    fileNamePrefix = EXPORT_DESC,
    fileFormat     = 'CSV'
)
export_task.start()

print(f'Task ID : {export_task.id}')
print(f'Monitor : https://code.earthengine.google.com/tasks')
print(f'Output  : Drive/{EXPORT_FOLDER}/{EXPORT_DESC}.csv')

In [ ]:
# ==========================================
# 5. POLL UNTIL EXPORT COMPLETES
# ==========================================
# Unchanged from original.

print('Waiting for export to complete...')
print('(Checks every 30 seconds. Safe to interrupt.)')

while True:
    status = export_task.status()
    state  = status['state']
    print(f"  [{time.strftime('%H:%M:%S')}] Status: {state}")

    if state == 'COMPLETED':
        print('\n\u2713 Export complete.')
        break
    elif state in ('FAILED', 'CANCELLED'):
        raise RuntimeError(
            f"GEE export failed: "
            f"{status.get('error_message', state)}")
    else:
        time.sleep(30)

In [ ]:
# ==========================================
# 6. LOAD EXPORTED CSV AND CLEAN
# ==========================================
# CHANGED:
#   - Reads PointID instead of DHSCLUST
#   - Renames 'median' -> 'VIIRS_Median' to match the
#     training static feature column name exactly
#   - NTL_Class computation REMOVED (not needed for inference)

raw_csv = os.path.join(OUTPUT_DIR, f'{EXPORT_DESC}.csv')
time.sleep(5)   # Let Drive sync

if not os.path.exists(raw_csv):
    raise FileNotFoundError(
        f'CSV not found at {raw_csv}. '
        'Wait a moment for Drive to sync then rerun this cell.')

df_raw = pd.read_csv(raw_csv)
print(f'Loaded {len(df_raw)} rows')
print(f'Columns: {list(df_raw.columns)}')
print(f'\nSample:')
print(df_raw.head())

# Rename to match training schema
df = df_raw.rename(columns={'median': 'VIIRS_Median'}).copy()
df['PointID'] = df['PointID'].astype(int)

# Handle missing values
n_missing = df['VIIRS_Median'].isna().sum()
print(f'\nMissing NTL values: {n_missing} points')
if n_missing > 0:
    print('Points with missing VIIRS:')
    print(df[df['VIIRS_Median'].isna()]
          [['PointID', 'Province', 'Municipality']])
    print('Filling with 0 (dark/conservative).')
    df['VIIRS_Median'] = df['VIIRS_Median'].fillna(0)

print(f'\nVIIRS_Median statistics:')
print(df['VIIRS_Median'].describe().round(4))
print(f'Zeros: {(df["VIIRS_Median"] == 0).sum()}')

In [ ]:
# ==========================================
# 7. VALIDATE COVERAGE AND SAVE
# ==========================================
# CHANGED:
#   - Cross-checks against prediction_points.csv by PointID
#   - No NTL_Class column in output
#   - Output: viirs_ntl_2025_prediction_points.csv

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR, 'viirs_ntl_2025_prediction_points.csv')

# Coverage check
original_ids = set(df_points['PointID'].astype(int))
exported_ids = set(df['PointID'].astype(int))
missing_ids  = original_ids - exported_ids

print(f'Original prediction points : {len(original_ids)}')
print(f'With VIIRS values          : {len(exported_ids)}')
print(f'Missing                    : {len(missing_ids)}')
if missing_ids:
    df_miss = df_points[
        df_points['PointID'].isin(missing_ids)]
    print(f'Missing by province:')
    print(df_miss['Province'].value_counts().to_string())

# Per-province summary
print(f'\nPer-province VIIRS_Median (nW/cm2/sr):')
prov = df.groupby('Province')['VIIRS_Median'].agg(
    Mean='mean', Min='min', Max='max', N='count'
).round(4)
print(prov.to_string())

# Sort and save
df_out = df.sort_values('PointID').reset_index(drop=True)
df_out.to_csv(OUTPUT_CSV, index=False)

print(f'\n{"="*50}')
print('PROCESSING SUMMARY')
print(f'{"="*50}')
print(f'Points processed : {len(df_out)}')
print(f'VIIRS year       : 2025')
print(f'NTL range        : [{df_out["VIIRS_Median"].min():.4f},'
      f' {df_out["VIIRS_Median"].max():.4f}] nW/cm2/sr')
print(f'Zeros            : {(df_out["VIIRS_Median"] == 0).sum()}')
print(f'Output columns   : {list(df_out.columns)}')
print(f'\nSaved to: {OUTPUT_CSV}')
print(f'\nNext: run phase3-osm-feature-extraction-prediction-points')
print(f'to generate OSM static features, then merge both.')